# Phase 4 — TF-IDF + Fine-Tuned MiniLM Reciprocal Rank Fusion

This notebook tests whether lexical and semantic retrieval are complementary.
It combines their independent rankings without training another model:

```text
query
  ├─ hybrid word/character TF-IDF -> Top-50
  └─ fine-tuned MiniLM            -> Top-50
                         ↓
               equal-weight RRF
                         ↓
                      Top-10
```

Development validation uses the Phase 1 TF-IDF fit on `dev_fit` and the Phase 3
development checkpoint trained on `dev_fit`. It never evaluates validation with
the final MiniLM retrained on full official train. After RRF configuration is
frozen, official test uses TF-IDF refit on full official train and the frozen
Phase 3 final checkpoint.

Default switches keep validation retrieval and official test disabled. This
notebook does not fine-tune a neural model.


In [3]:
import gc
import hashlib
import html
import json
import math
import platform
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
    import sentence_transformers
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"
    ])
    import torch
    import sentence_transformers
    from sentence_transformers import SentenceTransformer

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.12.13
PyTorch: 2.10.0+cu128
sentence-transformers: 5.4.1
CUDA available: True


## 1. Paths and frozen RRF search space

`COMPONENT_DEPTH` controls how many candidates each component contributes.
`RRF_C` is the reciprocal-rank constant; `OUTPUT_K` remains 10. Equal component
weights are fixed to avoid adding another tuning dimension.

The primary selection rule remains macro-project Hit@10, followed by MRR@10 and
Recall@10. Ties between RRF constants prefer the value closest to the conventional
starting point 60, then the smaller value.


In [4]:
# ===== KAGGLE PATHS =====

DATA_ROOT = Path(
    "/kaggle/input/datasets/nguyendinhtann/duplicatedetection/DuplicateDetection_Kaggle"
)

# Input của Kaggle là read-only
PHASE3_OUTPUT_ROOT = DATA_ROOT / "phase3_minilm_finetune_outputs"
PHASE4_INPUT_ROOT = DATA_ROOT / "phase4_rrf_outputs"

# File sinh mới phải ghi vào /kaggle/working
OUTPUT_ROOT = Path("/kaggle/working/phase4_rrf_outputs")
CACHE_ROOT = OUTPUT_ROOT / "cache"

DEV_DENSE_CHECKPOINT = PHASE3_OUTPUT_ROOT / "dev_model" / "final"
FINAL_DENSE_CHECKPOINT = PHASE3_OUTPUT_ROOT / "final_model" / "final"

for path in (OUTPUT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

# Copy kết quả validation đã freeze (c=10) sang working
import shutil

shutil.copy2(
    PHASE4_INPUT_ROOT / "phase4_rrf_selection.json",
    OUTPUT_ROOT / "phase4_rrf_selection.json",
)
DEV_DENSE_CHECKPOINT = PHASE3_OUTPUT_ROOT / "dev_model" / "final"
FINAL_DENSE_CHECKPOINT = PHASE3_OUTPUT_ROOT / "final_model" / "final"
for path in (OUTPUT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

PROJECTS = {
    "Firefox": {
        "main": DATA_ROOT / "Firefox" / "mozilla_firefox.csv",
        "train": DATA_ROOT / "Firefox" / "train.csv",
        "test": DATA_ROOT / "Firefox" / "test.csv",
    },
    "MozillaCore": {
        "main": DATA_ROOT / "MozillaCore" / "mozilla_core.csv",
        "train": DATA_ROOT / "MozillaCore" / "train.csv",
        "test": DATA_ROOT / "MozillaCore" / "test.csv",
    },
}

VALIDATION_FRACTION_OF_OFFICIAL_TRAIN = 0.15
METRIC_KS = (1, 5, 10)
TOP_KS = METRIC_KS  # Compatibility with the shared temporal-query code.
COMPONENT_DEPTH = 50
OUTPUT_K = 10
RRF_C_GRID = (10, 30, 60, 100)
TFIDF_WEIGHT = 1.0
DENSE_WEIGHT = 1.0

BASE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BASE_MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"
DESCRIPTION_CHAR_LIMIT = 4_000
EMBEDDING_TEXT_VERSION = f"title-newline-description-head-{DESCRIPTION_CHAR_LIMIT}"
ENCODE_BATCH_SIZE = 128
EVAL_QUERY_BATCH_SIZE = 64
EVAL_DEVICE = None

TFIDF_WORD_MAX_FEATURES = 200_000
TFIDF_CHAR_MAX_FEATURES = 100_000
TFIDF_WORD_WEIGHT = 0.80
TFIDF_CHAR_WEIGHT = 0.20

SELECTION_AGGREGATION = "macro_project_average"
PRIMARY_SELECTION_METRIC = "hit@10"
SELECTION_TIE_BREAKERS = ("mrr@10", "recall@10")

RUN_VALIDATION_COMPONENT_RETRIEVAL = False
RUN_RRF_VALIDATION = False
RUN_OFFICIAL_TEST = True
FORCE_REBUILD_RANKINGS = False
FORCE_REBUILD_EMBEDDINGS = False

for project, paths in PROJECTS.items():
    for kind, path in paths.items():
        assert path.exists(), f"Missing {project}/{kind}: {path}"
assert COMPONENT_DEPTH >= OUTPUT_K >= max(METRIC_KS)
assert all(value > 0 for value in RRF_C_GRID)
assert len(set(RRF_C_GRID)) == len(RRF_C_GRID)
assert TFIDF_WEIGHT > 0 and DENSE_WEIGHT > 0

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("Component depth / output K:", COMPONENT_DEPTH, OUTPUT_K)
print("RRF_C grid:", RRF_C_GRID)
print("Validation retrieval / RRF validation / official test:",
      RUN_VALIDATION_COMPONENT_RETRIEVAL, RUN_RRF_VALIDATION, RUN_OFFICIAL_TEST)

print("DATA_ROOT:", DATA_ROOT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

print("Firefox:", PROJECTS["Firefox"]["main"].exists())
print("MozillaCore:", PROJECTS["MozillaCore"]["main"].exists())
print("Final model:", FINAL_DENSE_CHECKPOINT.exists())
print(
    "Phase 3 selection:",
    (PHASE3_OUTPUT_ROOT / "phase3_model_selection.json").exists()
)
print(
    "Phase 4 selection:",
    (OUTPUT_ROOT / "phase4_rrf_selection.json").exists()
)


DATA_ROOT: /kaggle/input/datasets/nguyendinhtann/duplicatedetection/DuplicateDetection_Kaggle
OUTPUT_ROOT: /kaggle/working/phase4_rrf_outputs
Component depth / output K: 50 10
RRF_C grid: (10, 30, 60, 100)
Validation retrieval / RRF validation / official test: False False True
DATA_ROOT: /kaggle/input/datasets/nguyendinhtann/duplicatedetection/DuplicateDetection_Kaggle
GPU: Tesla T4
Firefox: True
MozillaCore: True
Final model: True
Phase 3 selection: True
Phase 4 selection: True


## 2. Load and normalize issues and official relevance files

`Duplicate` is a semicolon-separated multi-relevant set and is never reduced to
one target. Phase 2 keeps Phase 1's text normalization for identical data gates.
For the encoder, `embedding_text` contains the title once followed by the head of
the description; this avoids the Phase 1 TF-IDF title repetition and makes the
model's truncation policy explicit.


In [5]:
def normalize_id(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return re.sub(r"\.0$", "", text)


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " <URL> ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_duplicate_set(value):
    if pd.isna(value):
        return frozenset()
    text = str(value).strip()
    if not text or text.upper() == "NULL":
        return frozenset()
    return frozenset(
        normalized
        for item in text.split(";")
        if (normalized := normalize_id(item))
    )


def load_project(project, paths):
    issues = pd.read_csv(paths["main"], low_memory=False)
    issues = issues.rename(columns={
        "Issue_id": "issue_id",
        "Priority": "priority",
        "Component": "component",
        "Duplicated_issue": "duplicate_of",
        "Title": "title",
        "Description": "description",
        "Resolution": "resolution",
        "Created_time": "created_at",
    })
    required = {
        "issue_id", "title", "description", "created_at",
        "duplicate_of", "component", "resolution",
    }
    assert required.issubset(issues.columns), required - set(issues.columns)

    issues["issue_id"] = issues["issue_id"].map(normalize_id)
    issues["created_at"] = pd.to_datetime(issues["created_at"], errors="coerce", utc=True)
    issues["title_clean"] = issues["title"].map(normalize_text)
    issues["description_clean"] = issues["description"].map(normalize_text)

    # Keep the Phase 1 field for identical non-empty-text filtering.
    issues["text"] = (
        issues["title_clean"] + " . " + issues["title_clean"]
        + " . " + issues["description_clean"]
    ).str.strip()

    description_head = issues["description_clean"].str.slice(0, DESCRIPTION_CHAR_LIMIT)
    issues["embedding_text"] = (
        issues["title_clean"] + "\n" + description_head
    ).str.strip()
    issues["description_char_truncated"] = (
        issues["description_clean"].str.len() > DESCRIPTION_CHAR_LIMIT
    )
    issues["project"] = project

    assert issues["issue_id"].notna().all()
    assert not issues["issue_id"].duplicated().any()
    assert issues["created_at"].notna().all()
    issues = issues.sort_values(["created_at", "issue_id"]).reset_index(drop=True)
    issues["position"] = np.arange(len(issues), dtype=np.int64)

    split_frames = {}
    for split in ("train", "test"):
        labels = pd.read_csv(paths[split], dtype=str)
        assert {"Issue_id", "Duplicate"}.issubset(labels.columns)
        labels = labels.rename(columns={"Issue_id": "issue_id", "Duplicate": "duplicate_raw"})
        labels["issue_id"] = labels["issue_id"].map(normalize_id)
        labels["relevant_ids"] = labels["duplicate_raw"].map(parse_duplicate_set)
        assert labels["issue_id"].notna().all()
        assert not labels["issue_id"].duplicated().any()
        split_frames[split] = labels[["issue_id", "relevant_ids"]].copy()

    all_ids = set(issues["issue_id"])
    train_ids = set(split_frames["train"]["issue_id"])
    test_ids = set(split_frames["test"]["issue_id"])
    assert train_ids.isdisjoint(test_ids), f"{project}: train/test ID overlap"
    assert train_ids | test_ids == all_ids, f"{project}: split does not cover main CSV"

    id_to_time = issues.set_index("issue_id")["created_at"].to_dict()
    id_to_text = issues.set_index("issue_id")["text"].to_dict()
    known = set(id_to_time)

    audit_rows = []
    for split, labels in split_frames.items():
        total_links = sum(len(x) for x in labels["relevant_ids"])
        missing = sum(
            target not in known
            for targets in labels["relevant_ids"]
            for target in targets
        )
        future = same = 0
        for query_id, targets in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
            query_time = id_to_time[query_id]
            for target in targets:
                if target not in id_to_time:
                    continue
                future += id_to_time[target] > query_time
                same += id_to_time[target] == query_time
        audit_rows.append({
            "project": project,
            "split": split,
            "rows": len(labels),
            "labeled_queries": int((labels["relevant_ids"].map(len) > 0).sum()),
            "relevance_links": total_links,
            "missing_targets": missing,
            "future_targets": future,
            "same_time_targets": same,
            "description_char_truncated_pct": round(
                100 * issues["description_char_truncated"].mean(), 2
            ),
        })

    return {
        "project": project,
        "issues": issues,
        "train_labels": split_frames["train"],
        "test_labels": split_frames["test"],
        "id_to_time": id_to_time,
        "id_to_text": id_to_text,
        "audit": pd.DataFrame(audit_rows),
    }


datasets = {
    project: load_project(project, paths)
    for project, paths in PROJECTS.items()
}

data_audit = pd.concat([data["audit"] for data in datasets.values()], ignore_index=True)
display(data_audit)


,project,split,rows,labeled_queries,relevance_links,missing_targets,future_targets,same_time_targets,description_char_truncated_pct
0,Firefox,train,92651,30026,310378,0,155224,22,0.87
1,Firefox,test,23163,4236,109926,0,0,2,0.87
2,MozillaCore,train,164055,50122,281511,0,140798,144,2.10
3,MozillaCore,test,41014,4115,8937,0,0,1,2.10


## 3. Data-gate assertions

The official test files are suitable for rolling temporal retrieval only when
all labeled targets exist and none is from the future. Training label files may
contain future cluster members; validation construction filters those targets
at each query timestamp.


In [6]:
dataset_summary = []
for project, data in datasets.items():
    issues = data["issues"]
    test_audit = data["audit"].query("split == 'test'").iloc[0]
    assert test_audit["missing_targets"] == 0
    assert test_audit["future_targets"] == 0
    assert not issues["issue_id"].duplicated().any()
    dataset_summary.append({
        "project": project,
        "issues": len(issues),
        "empty_text": int(issues["text"].str.len().eq(0).sum()),
        "created_min": issues["created_at"].min(),
        "created_max": issues["created_at"].max(),
        "official_train": len(data["train_labels"]),
        "official_test": len(data["test_labels"]),
        "test_labeled_queries": int(test_audit["labeled_queries"]),
        "test_relevance_links": int(test_audit["relevance_links"]),
    })

dataset_summary = pd.DataFrame(dataset_summary)
display(dataset_summary)
print("DATA GATE: PASS")


# Split-level leakage guard. Validation may use only official-train candidates.
# The current BugHub files are chronological, but the assertion makes that
# assumption executable instead of relying on the README or filename.
split_chronology_rows = []
for project, data in datasets.items():
    issues = data["issues"]
    official_train_ids = set(data["train_labels"]["issue_id"])
    official_test_ids = set(data["test_labels"]["issue_id"])
    train_times = issues.loc[
        issues["issue_id"].isin(official_train_ids), "created_at"
    ]
    test_times = issues.loc[
        issues["issue_id"].isin(official_test_ids), "created_at"
    ]
    train_max = train_times.max()
    test_min = test_times.min()
    assert train_max < test_min, (
        f"{project}: official split is not strictly chronological: "
        f"train_max={train_max}, test_min={test_min}. "
        "Keep the explicit allowed-candidate mask enabled."
    )
    data["official_train_ids"] = official_train_ids
    data["official_test_ids"] = official_test_ids
    split_chronology_rows.append({
        "project": project,
        "train_min": train_times.min(),
        "train_max": train_max,
        "test_min": test_min,
        "test_max": test_times.max(),
        "strictly_chronological": True,
    })

split_chronology = pd.DataFrame(split_chronology_rows)
display(split_chronology)
print("SPLIT CHRONOLOGY GATE: PASS")


,project,issues,empty_text,created_min,created_max,official_train,official_test,test_labeled_queries,test_relevance_links
0,Firefox,115814,0,1999-07-30 22:55:51+00:00,2014-01-01 06:10:17+00:00,92651,23163,4236,109926
1,MozillaCore,205069,0,1997-03-28 16:00:00+00:00,2013-12-31 20:02:55+00:00,164055,41014,4115,8937


DATA GATE: PASS


,project,train_min,train_max,test_min,test_max,strictly_chronological
0,Firefox,1999-07-30 22:55:51+00:00,2011-04-05 07:40:37+00:00,2011-04-05 08:00:44+00:00,2014-01-01 06:10:17+00:00,True
1,MozillaCore,1997-03-28 16:00:00+00:00,2011-07-24 19:14:06+00:00,2011-07-24 20:15:38+00:00,2013-12-31 20:02:55+00:00,True


SPLIT CHRONOLOGY GATE: PASS


## 4. Construct validation and test query sets

For every query, relevant reports are filtered to IDs that:

- exist in the same project corpus;
- have usable text;
- were created strictly before the query.

Queries with no remaining historical relevant report are not evaluable and are
reported separately. This prevents future-target leakage.


In [7]:
def filter_temporal_queries(data, labels, allowed_query_ids=None):
    id_to_time = data["id_to_time"]
    id_to_text = data["id_to_text"]
    records = []
    audit = {
        "labeled_queries": 0,
        "empty_query_text": 0,
        "no_historical_relevant": 0,
        "future_or_same_relevant_removed": 0,
        "missing_or_empty_relevant_removed": 0,
    }
    allowed_query_ids = None if allowed_query_ids is None else set(allowed_query_ids)

    for query_id, raw_relevant in labels[["issue_id", "relevant_ids"]].itertuples(index=False):
        if allowed_query_ids is not None and query_id not in allowed_query_ids:
            continue
        if not raw_relevant:
            continue
        audit["labeled_queries"] += 1
        if not id_to_text.get(query_id, ""):
            audit["empty_query_text"] += 1
            continue

        query_time = id_to_time[query_id]
        relevant = set()
        for target in raw_relevant:
            if target not in id_to_time or not id_to_text.get(target, ""):
                audit["missing_or_empty_relevant_removed"] += 1
                continue
            if id_to_time[target] >= query_time:
                audit["future_or_same_relevant_removed"] += 1
                continue
            relevant.add(target)

        if not relevant:
            audit["no_historical_relevant"] += 1
            continue
        records.append({
            "query_id": query_id,
            "query_time": query_time,
            "relevant_ids": frozenset(relevant),
            "num_relevant": len(relevant),
        })

    query_frame = pd.DataFrame(records).sort_values(["query_time", "query_id"]).reset_index(drop=True)
    audit["evaluable_queries"] = len(query_frame)
    audit["evaluable_relevance_links"] = int(query_frame["num_relevant"].sum()) if len(query_frame) else 0
    return query_frame, audit


query_sets = {}
query_audits = []
for project, data in datasets.items():
    issues = data["issues"]
    train_labels = data["train_labels"]
    official_train_ids = set(train_labels["issue_id"])
    train_issues = issues[issues["issue_id"].isin(official_train_ids)].sort_values(["created_at", "issue_id"])

    validation_size = math.ceil(len(train_issues) * VALIDATION_FRACTION_OF_OFFICIAL_TRAIN)
    validation_ids = set(train_issues.tail(validation_size)["issue_id"])
    dev_fit_ids = set(train_issues.iloc[:-validation_size]["issue_id"])

    validation_queries, validation_audit = filter_temporal_queries(
        data, train_labels, validation_ids
    )
    test_queries, test_audit = filter_temporal_queries(data, data["test_labels"])

    assert all(
        data["id_to_time"][target] < row.query_time
        for row in validation_queries.itertuples()
        for target in row.relevant_ids
    )
    assert all(
        data["id_to_time"][target] < row.query_time
        for row in test_queries.itertuples()
        for target in row.relevant_ids
    )

    query_sets[project] = {
        "dev_fit_ids": dev_fit_ids,
        "official_train_ids": official_train_ids,
        "validation": validation_queries,
        "test": test_queries,
    }
    for split, audit in (("validation", validation_audit), ("test", test_audit)):
        query_audits.append({"project": project, "split": split, **audit})

query_audits = pd.DataFrame(query_audits)
display(query_audits)
print("TEMPORAL GROUND TRUTH: PASS")


,project,split,labeled_queries,empty_query_text,no_historical_relevant,future_or_same_relevant_removed,missing_or_empty_relevant_removed,evaluable_queries,evaluable_relevance_links
0,Firefox,validation,3986,0,926,9131,0,3060,17577
1,Firefox,test,4236,0,0,2,0,4236,109924
2,MozillaCore,validation,5071,0,1714,4636,0,3357,7524
3,MozillaCore,test,4115,0,0,1,0,4115,8936


TEMPORAL GROUND TRUTH: PASS


## 5. Component ranking engines and common metrics

Both component retrievers receive the same query table and explicit candidate
mask. Validation candidates are official-train reports older than the query.
Official-test candidates are official-train plus earlier official-test reports.

TF-IDF reproduces Phase 1's selected hybrid word/character representation. Dense
retrieval uses normalized embeddings from the Phase 3 checkpoint. Both produce
Top-50 rankings; metrics are still evaluated only at 1/5/10.


In [8]:
def ranking_metrics(retrieved_ids, relevant_ids, ks=METRIC_KS):
    relevant = set(relevant_ids)
    result = {}
    for k in ks:
        hits = len(set(retrieved_ids[:k]) & relevant)
        result[f"hit@{k}"] = float(hits > 0)
        result[f"recall@{k}"] = hits / len(relevant)
    first_rank = next(
        (rank for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
         if issue_id in relevant),
        None,
    )
    result["mrr@10"] = 0.0 if first_rank is None else 1.0 / first_rank
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, issue_id in enumerate(retrieved_ids[:10], start=1)
        if issue_id in relevant
    )
    ideal_hits = min(len(relevant), 10)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    result["ndcg@10"] = dcg / idcg if idcg else 0.0
    return result


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return np.array([], dtype=np.int64)
    if k == len(scores):
        candidates = np.arange(len(scores))
    else:
        candidates = np.argpartition(-scores, k - 1)[:k]
    order = np.lexsort((candidates, -scores[candidates]))
    return candidates[order]


def fit_hybrid_tfidf(issues, fit_ids):
    fit_rows = issues[issues["issue_id"].isin(set(fit_ids))]
    assert len(fit_rows) > 0
    word = TfidfVectorizer(
        lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
        min_df=2, max_df=0.995, max_features=TFIDF_WORD_MAX_FEATURES,
        sublinear_tf=True, dtype=np.float32,
    )
    word.fit(fit_rows["text"])
    word_matrix = normalize(word.transform(issues["text"]), copy=False)
    char = TfidfVectorizer(
        analyzer="char_wb", lowercase=True, ngram_range=(3, 5), min_df=3,
        max_features=TFIDF_CHAR_MAX_FEATURES, sublinear_tf=True,
        dtype=np.float32,
    )
    char.fit(fit_rows["title_clean"])
    char_matrix = normalize(char.transform(issues["title_clean"]), copy=False)
    combined = hstack([
        word_matrix * np.float32(math.sqrt(TFIDF_WORD_WEIGHT)),
        char_matrix * np.float32(math.sqrt(TFIDF_CHAR_WEIGHT)),
    ], format="csr", dtype=np.float32)
    combined = normalize(combined, copy=False)
    metadata = {
        "word_vocabulary": len(word.vocabulary_),
        "char_vocabulary": len(char.vocabulary_),
    }
    del word_matrix, char_matrix
    gc.collect()
    return combined, metadata


def candidate_mask(issue_ids, allowed_candidate_ids):
    allowed = set(allowed_candidate_ids)
    mask = np.fromiter(
        (issue_id in allowed for issue_id in issue_ids),
        dtype=bool, count=len(issue_ids),
    )
    assert mask.any()
    return mask, allowed


def legal_top_positions(scores, allowed_mask, candidate_end, depth):
    local_allowed = allowed_mask[:candidate_end]
    if local_allowed.all():
        return top_k_indices(scores[:candidate_end], depth)
    legal_positions = np.flatnonzero(local_allowed)
    legal_top = top_k_indices(scores[legal_positions], depth)
    return legal_positions[legal_top]


def retrieve_sparse_rolling(data, queries, matrix, allowed_candidate_ids, depth):
    issues = data["issues"]
    issue_ids = issues["issue_id"].to_numpy()
    created_ns = issues["created_at"].astype("int64").to_numpy()
    id_to_pos = dict(zip(issue_ids, issues["position"]))
    allowed_mask, allowed = candidate_mask(issue_ids, allowed_candidate_ids)
    rows, started = [], time.time()
    for number, row in enumerate(queries.itertuples(index=False), start=1):
        query_pos = id_to_pos[row.query_id]
        candidate_end = int(np.searchsorted(
            created_ns, int(row.query_time.value), side="left"
        ))
        scores = (matrix[:candidate_end] @ matrix[query_pos].T).toarray().ravel()
        top_positions = legal_top_positions(scores, allowed_mask, candidate_end, depth)
        retrieved = issue_ids[top_positions].tolist()
        assert row.query_id not in retrieved
        assert set(retrieved).issubset(allowed)
        assert all(data["id_to_time"][item] < row.query_time for item in retrieved)
        rows.append({
            "project": data["project"], "query_id": row.query_id,
            "query_time": row.query_time,
            "num_candidates": int(allowed_mask[:candidate_end].sum()),
            "retrieved_ids": ";".join(retrieved),
        })
        if number % 500 == 0:
            print(f"  {data['project']} / TF-IDF: {number}/{len(queries)}")
    print("TF-IDF retrieval seconds:", round(time.time() - started, 2))
    return pd.DataFrame(rows)


def checkpoint_fingerprint(model_path):
    model_path = Path(model_path)
    digest, included = hashlib.sha256(), []
    for path in sorted(model_path.rglob("*")):
        if not path.is_file() or path.suffix not in {".safetensors", ".bin", ".json", ".txt"}:
            continue
        relative = path.relative_to(model_path).as_posix()
        included.append(relative)
        digest.update(relative.encode("utf-8"))
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
    assert included, f"No checkpoint files found under {model_path}"
    return digest.hexdigest()


def encode_checkpoint_for_project(model, data, model_tag, fingerprint):
    issues = data["issues"]
    digest = hashlib.sha256()
    digest.update(fingerprint.encode("utf-8"))
    digest.update(EMBEDDING_TEXT_VERSION.encode("utf-8"))
    for issue_id, text in issues[["issue_id", "embedding_text"]].itertuples(index=False):
        digest.update(issue_id.encode("utf-8"))
        digest.update(b"\0")
        digest.update(text.encode("utf-8"))
    path = CACHE_ROOT / f"{data['project'].lower()}__{model_tag}__{digest.hexdigest()[:20]}.npy"
    if path.exists() and not FORCE_REBUILD_EMBEDDINGS:
        matrix = np.load(path)
        assert matrix.shape[0] == len(issues)
        print("EMBEDDING CACHE HIT:", path.name)
        return matrix
    matrix = model.encode(
        issues["embedding_text"].tolist(), batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32, copy=False)
    np.save(path, matrix)
    return matrix


def retrieve_dense_rolling(data, queries, embeddings, allowed_candidate_ids, depth):
    issues = data["issues"]
    issue_ids = issues["issue_id"].to_numpy()
    created_ns = issues["created_at"].astype("int64").to_numpy()
    id_to_pos = dict(zip(issue_ids, issues["position"]))
    query_positions = np.asarray(
        [id_to_pos[item] for item in queries["query_id"]], dtype=np.int64
    )
    query_times_ns = queries["query_time"].astype("int64").to_numpy()
    candidate_ends = np.searchsorted(created_ns, query_times_ns, side="left")
    assert np.all(candidate_ends > 0)
    assert np.all(candidate_ends <= query_positions)
    allowed_mask, allowed = candidate_mask(issue_ids, allowed_candidate_ids)
    device = EVAL_DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
    corpus = torch.from_numpy(np.asarray(embeddings)).to(device)
    rows, started = [], time.time()
    with torch.inference_mode():
        for start in range(0, len(queries), EVAL_QUERY_BATCH_SIZE):
            stop = min(start + EVAL_QUERY_BATCH_SIZE, len(queries))
            positions = query_positions[start:stop]
            ends = candidate_ends[start:stop]
            corpus_end = int(ends.max())
            query_tensor = corpus[
                torch.as_tensor(positions, dtype=torch.long, device=device)
            ]
            score_batch = (query_tensor @ corpus[:corpus_end].T).float().cpu().numpy()
            for offset, row in enumerate(queries.iloc[start:stop].itertuples(index=False)):
                candidate_end = int(ends[offset])
                top_positions = legal_top_positions(
                    score_batch[offset], allowed_mask, candidate_end, depth
                )
                retrieved = issue_ids[top_positions].tolist()
                assert row.query_id not in retrieved
                assert set(retrieved).issubset(allowed)
                assert all(data["id_to_time"][item] < row.query_time for item in retrieved)
                rows.append({
                    "project": data["project"], "query_id": row.query_id,
                    "query_time": row.query_time,
                    "num_candidates": int(allowed_mask[:candidate_end].sum()),
                    "retrieved_ids": ";".join(retrieved),
                })
            if stop % 256 == 0 or stop == len(queries):
                print(f"  {data['project']} / dense: {stop}/{len(queries)}")
            del query_tensor, score_batch
    print("Dense retrieval seconds:", round(time.time() - started, 2))
    del corpus
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


def artifact_paths(split, project, system):
    stem = f"{project.lower()}_{split}_{system}_top{COMPONENT_DEPTH}"
    return OUTPUT_ROOT / f"{stem}.csv", OUTPUT_ROOT / f"{stem}.json"


def save_ranking_artifact(frame, split, project, system, metadata):
    csv_path, json_path = artifact_paths(split, project, system)
    frame.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")


def load_ranking_artifact(split, project, system, expected_metadata):
    csv_path, json_path = artifact_paths(split, project, system)
    if not csv_path.exists() or not json_path.exists():
        return None
    observed = json.loads(json_path.read_text(encoding="utf-8"))
    if any(observed.get(key) != value for key, value in expected_metadata.items()):
        return None
    frame = pd.read_csv(csv_path, dtype={"query_id": str, "retrieved_ids": str})
    return frame


metric_test = ranking_metrics(["x", "b", "y"], {"a", "b"})
assert metric_test["hit@1"] == 0 and metric_test["mrr@10"] == 0.5
assert top_k_indices(np.array([0.1, 0.9, 0.9]), 2).tolist() == [1, 2]
print("RANKING ENGINE TESTS: PASS")


RANKING ENGINE TESTS: PASS


## 6. Generate/load validation component Top-50 rankings

The TF-IDF vocabulary/IDF is fit only on `dev_fit`. The dense checkpoint must be
the Phase 3 development model trained only on `dev_fit`. Ranking artifacts carry
metadata and are rejected when the checkpoint, split, or depth changes.


In [9]:
validation_rankings = {}

if RUN_VALIDATION_COMPONENT_RETRIEVAL:
    dev_manifest_path = PHASE3_OUTPUT_ROOT / "dev_model" / "training_manifest.json"
    assert DEV_DENSE_CHECKPOINT.exists() and dev_manifest_path.exists(), (
        "Missing Phase 3 dev checkpoint."
    )
    dev_manifest = json.loads(dev_manifest_path.read_text(encoding="utf-8"))
    assert dev_manifest["scope"] == "dev_fit"
    assert dev_manifest["used_official_test_pairs"] is False
    dense_fingerprint = checkpoint_fingerprint(DEV_DENSE_CHECKPOINT)
    dense_model = SentenceTransformer(str(DEV_DENSE_CHECKPOINT))

    for project, data in datasets.items():
        print(f"\n=== VALIDATION COMPONENT RANKINGS: {project} ===")
        queries = query_sets[project]["validation"]
        allowed_ids = data["official_train_ids"]
        common = {
            "split": "validation", "project": project,
            "queries": len(queries), "component_depth": COMPONENT_DEPTH,
            "candidate_policy": "official_train_and_strictly_older",
            "embedding_text_version": EMBEDDING_TEXT_VERSION,
        }

        tfidf_metadata = {
            **common, "system": "hybrid_tfidf",
            "fit_scope": "dev_fit",
            "word_max_features": TFIDF_WORD_MAX_FEATURES,
            "char_max_features": TFIDF_CHAR_MAX_FEATURES,
            "word_weight": TFIDF_WORD_WEIGHT, "char_weight": TFIDF_CHAR_WEIGHT,
        }
        tfidf_rankings = None if FORCE_REBUILD_RANKINGS else load_ranking_artifact(
            "validation", project, "tfidf", tfidf_metadata
        )
        if tfidf_rankings is None:
            matrix, vocab_metadata = fit_hybrid_tfidf(
                data["issues"], query_sets[project]["dev_fit_ids"]
            )
            tfidf_rankings = retrieve_sparse_rolling(
                data, queries, matrix, allowed_ids, COMPONENT_DEPTH
            )
            save_ranking_artifact(
                tfidf_rankings, "validation", project, "tfidf",
                {**tfidf_metadata, **vocab_metadata},
            )
            # Reload expects the complete metadata written above.
            tfidf_metadata = {**tfidf_metadata, **vocab_metadata}
            del matrix
            gc.collect()

        dense_metadata = {
            **common, "system": "fine_tuned_minilm",
            "fit_scope": "dev_fit", "checkpoint_fingerprint": dense_fingerprint,
        }
        dense_rankings = None if FORCE_REBUILD_RANKINGS else load_ranking_artifact(
            "validation", project, "dense", dense_metadata
        )
        if dense_rankings is None:
            embeddings = encode_checkpoint_for_project(
                dense_model, data, "dev_finetuned", dense_fingerprint
            )
            dense_rankings = retrieve_dense_rolling(
                data, queries, embeddings, allowed_ids, COMPONENT_DEPTH
            )
            save_ranking_artifact(
                dense_rankings, "validation", project, "dense", dense_metadata
            )
        validation_rankings[(project, "tfidf")] = tfidf_rankings
        validation_rankings[(project, "dense")] = dense_rankings
else:
    print("VALIDATION COMPONENT RETRIEVAL LOCKED.")


VALIDATION COMPONENT RETRIEVAL LOCKED.


## 7. RRF validation, complementarity, and selection

RRF score for document `d`:

```text
TFIDF_WEIGHT / (RRF_C + tfidf_rank(d))
+ DENSE_WEIGHT / (RRF_C + dense_rank(d))
```

Missing documents contribute zero. Ties are deterministic: RRF score descending,
best component rank ascending, then chronological corpus position ascending.
The notebook reports component metrics, every RRF candidate, paired outcomes, and
Top-50 union coverage before freezing the winner.


In [10]:
def split_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [item for item in str(value).split(";") if item]


def rrf_fuse(tfidf_ids, dense_ids, rrf_c, issue_position, output_k=OUTPUT_K):
    scores, best_rank = {}, {}
    for weight, ranking in ((TFIDF_WEIGHT, tfidf_ids), (DENSE_WEIGHT, dense_ids)):
        for rank, issue_id in enumerate(ranking[:COMPONENT_DEPTH], start=1):
            scores[issue_id] = scores.get(issue_id, 0.0) + weight / (rrf_c + rank)
            best_rank[issue_id] = min(best_rank.get(issue_id, rank), rank)
    ordered = sorted(
        scores,
        key=lambda issue_id: (
            -scores[issue_id], best_rank[issue_id], issue_position[issue_id]
        ),
    )
    return ordered[:output_k]


def score_component(frame, query_frame, system):
    relevant_map = query_frame.set_index("query_id")["relevant_ids"].to_dict()
    rows = []
    for row in frame.itertuples(index=False):
        ranking = split_ids(row.retrieved_ids)
        rows.append({
            "project": row.project, "query_id": row.query_id, "system": system,
            **ranking_metrics(ranking, relevant_map[row.query_id]),
        })
    return pd.DataFrame(rows)


def fuse_project(tfidf_frame, dense_frame, query_frame, data, rrf_c):
    assert set(tfidf_frame["query_id"]) == set(query_frame["query_id"])
    assert set(dense_frame["query_id"]) == set(query_frame["query_id"])
    merged = tfidf_frame[["query_id", "retrieved_ids"]].merge(
        dense_frame[["query_id", "retrieved_ids"]], on="query_id",
        suffixes=("_tfidf", "_dense"), validate="one_to_one",
    )
    relevant_map = query_frame.set_index("query_id")["relevant_ids"].to_dict()
    position = data["issues"].set_index("issue_id")["position"].to_dict()
    rows = []
    for row in merged.itertuples(index=False):
        tfidf_ids = split_ids(row.retrieved_ids_tfidf)
        dense_ids = split_ids(row.retrieved_ids_dense)
        fused = rrf_fuse(tfidf_ids, dense_ids, rrf_c, position)
        relevant = relevant_map[row.query_id]
        union_ids = set(tfidf_ids[:COMPONENT_DEPTH]) | set(dense_ids[:COMPONENT_DEPTH])
        rows.append({
            "project": data["project"], "query_id": row.query_id,
            "rrf_c": rrf_c, "retrieved_ids": ";".join(fused),
            "union_hit@50": float(bool(union_ids & set(relevant))),
            **ranking_metrics(fused, relevant),
        })
    return pd.DataFrame(rows)


def summarize(frame, system, rrf_c=None):
    metrics = [
        *(f"hit@{k}" for k in METRIC_KS), *(f"recall@{k}" for k in METRIC_KS),
        "mrr@10", "ndcg@10",
    ]
    return {
        "project": frame["project"].iloc[0], "system": system,
        "rrf_c": rrf_c, "queries": len(frame),
        **{metric: frame[metric].mean() for metric in metrics},
        "union_hit@50": frame["union_hit@50"].mean()
            if "union_hit@50" in frame else np.nan,
    }


def macro_row(project_rows, system, rrf_c=None):
    metrics = [
        *(f"hit@{k}" for k in METRIC_KS), *(f"recall@{k}" for k in METRIC_KS),
        "mrr@10", "ndcg@10",
    ]
    return {
        "project": "MACRO_PROJECT_AVERAGE", "system": system,
        "rrf_c": rrf_c, "queries": int(project_rows["queries"].sum()),
        **{metric: project_rows[metric].mean() for metric in metrics},
        "union_hit@50": project_rows["union_hit@50"].mean()
            if project_rows["union_hit@50"].notna().any() else np.nan,
    }


def selection_tuple(row):
    return tuple(float(row[metric]) for metric in (
        PRIMARY_SELECTION_METRIC, *SELECTION_TIE_BREAKERS
    ))


rrf_validation_summary = pd.DataFrame()
paired_complementarity = pd.DataFrame()
phase4_selection = None

if RUN_RRF_VALIDATION:
    if not validation_rankings:
        raise RuntimeError(
            "Run Section 6 first to create/load Top-50 component rankings."
        )
    all_summary_rows, fused_by_c = [], {}
    component_query_results = {}
    for project, data in datasets.items():
        queries = query_sets[project]["validation"]
        tfidf_scored = score_component(
            validation_rankings[(project, "tfidf")], queries, "tfidf"
        )
        dense_scored = score_component(
            validation_rankings[(project, "dense")], queries, "fine_tuned_dense"
        )
        component_query_results[(project, "tfidf")] = tfidf_scored
        component_query_results[(project, "dense")] = dense_scored
        all_summary_rows.extend([
            summarize(tfidf_scored, "tfidf"),
            summarize(dense_scored, "fine_tuned_dense"),
        ])
        for rrf_c in RRF_C_GRID:
            fused = fuse_project(
                validation_rankings[(project, "tfidf")],
                validation_rankings[(project, "dense")],
                queries, data, rrf_c,
            )
            fused_by_c[(project, rrf_c)] = fused
            all_summary_rows.append(summarize(fused, "rrf_hybrid", rrf_c))

    project_summary = pd.DataFrame(all_summary_rows)
    macro_rows = []
    for (system, rrf_c), group in project_summary.groupby(
        ["system", "rrf_c"], dropna=False, sort=True
    ):
        macro_rows.append(macro_row(group, system, rrf_c))
    rrf_validation_summary = pd.concat(
        [project_summary, pd.DataFrame(macro_rows)], ignore_index=True
    )
    display(rrf_validation_summary)
    rrf_validation_summary.to_csv(OUTPUT_ROOT / "rrf_validation_summary.csv", index=False)

    hybrid_macro = rrf_validation_summary.query(
        "project == 'MACRO_PROJECT_AVERAGE' and system == 'rrf_hybrid'"
    ).copy()
    hybrid_macro["distance_from_60"] = (hybrid_macro["rrf_c"] - 60).abs()
    ranking_metrics_order = [PRIMARY_SELECTION_METRIC, *SELECTION_TIE_BREAKERS]
    hybrid_macro = hybrid_macro.sort_values(
        [*ranking_metrics_order, "distance_from_60", "rrf_c"],
        ascending=([False] * len(ranking_metrics_order)) + [True, True],
        kind="mergesort",
    )
    selected = hybrid_macro.iloc[0]
    selected_c = int(selected["rrf_c"])

    component_macro = rrf_validation_summary.query(
        "project == 'MACRO_PROJECT_AVERAGE' and system != 'rrf_hybrid'"
    ).set_index("system")
    dense_score = selection_tuple(component_macro.loc["fine_tuned_dense"])
    tfidf_score = selection_tuple(component_macro.loc["tfidf"])
    hybrid_score = selection_tuple(selected)
    hybrid_wins = hybrid_score > max(dense_score, tfidf_score)

    paired_rows = []
    for project in PROJECTS:
        dense = component_query_results[(project, "dense")][["query_id", "hit@10"]]
        tfidf = component_query_results[(project, "tfidf")][["query_id", "hit@10"]]
        hybrid = fused_by_c[(project, selected_c)][["query_id", "hit@10", "union_hit@50"]]
        paired = dense.merge(
            tfidf, on="query_id", suffixes=("_dense", "_tfidf"), validate="one_to_one"
        ).merge(hybrid, on="query_id", validate="one_to_one")
        paired = paired.rename(columns={"hit@10": "hit@10_hybrid"})
        paired["project"] = project
        paired["dense_vs_tfidf"] = np.select(
            [
                (paired["hit@10_dense"] == 1) & (paired["hit@10_tfidf"] == 0),
                (paired["hit@10_dense"] == 0) & (paired["hit@10_tfidf"] == 1),
                (paired["hit@10_dense"] == 1) & (paired["hit@10_tfidf"] == 1),
            ], ["dense_only", "tfidf_only", "both_correct"], default="both_fail",
        )
        paired["hybrid_vs_dense"] = np.select(
            [
                (paired["hit@10_hybrid"] == 1) & (paired["hit@10_dense"] == 0),
                (paired["hit@10_hybrid"] == 0) & (paired["hit@10_dense"] == 1),
                (paired["hit@10_hybrid"] == 1) & (paired["hit@10_dense"] == 1),
            ], ["hybrid_win", "dense_win", "both_correct"], default="both_fail",
        )
        paired_rows.append(paired)
        fused_by_c[(project, selected_c)].to_csv(
            OUTPUT_ROOT / f"{project.lower()}_selected_rrf_validation_queries.csv",
            index=False,
        )
    paired_validation = pd.concat(paired_rows, ignore_index=True)
    paired_validation.to_csv(OUTPUT_ROOT / "paired_complementarity_queries.csv", index=False)
    paired_complementarity = pd.concat([
        paired_validation.groupby(["project", "dense_vs_tfidf"]).size()
            .rename("queries").reset_index().rename(columns={"dense_vs_tfidf": "outcome"})
            .assign(comparison="dense_vs_tfidf"),
        paired_validation.groupby(["project", "hybrid_vs_dense"]).size()
            .rename("queries").reset_index().rename(columns={"hybrid_vs_dense": "outcome"})
            .assign(comparison="hybrid_vs_dense"),
    ], ignore_index=True)
    display(paired_complementarity)
    paired_complementarity.to_csv(
        OUTPUT_ROOT / "paired_complementarity_summary.csv", index=False
    )

    phase4_selection = {
        "component_depth": COMPONENT_DEPTH,
        "output_k": OUTPUT_K,
        "rrf_c_grid": list(RRF_C_GRID),
        "selected_rrf_c": selected_c,
        "tfidf_weight": TFIDF_WEIGHT,
        "dense_weight": DENSE_WEIGHT,
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
        "hybrid_score_tuple": list(hybrid_score),
        "dense_score_tuple": list(dense_score),
        "tfidf_score_tuple": list(tfidf_score),
        "hybrid_beats_both_components": hybrid_wins,
        "full_validation": all(
            len(fused_by_c[(project, selected_c)]) == len(query_sets[project]["validation"])
            for project in PROJECTS
        ),
        "official_test_open_allowed": hybrid_wins,
    }
    (OUTPUT_ROOT / "phase4_rrf_selection.json").write_text(
        json.dumps(phase4_selection, indent=2), encoding="utf-8"
    )
    print(json.dumps(phase4_selection, indent=2))
else:
    print("RRF VALIDATION LOCKED.")


RRF VALIDATION LOCKED.


## 8. Frozen official test

The gate requires:

- hybrid beats both components on complete validation;
- Phase 3 selected fine-tuning and produced a final model trained on full official
  train without official-test pairs;
- component depth, RRF constant, weights, and metric rule exactly match the frozen
  Phase 4 selection artifact.

No test hyperparameter tuning is permitted.


In [11]:
official_test_summary = pd.DataFrame()

if RUN_OFFICIAL_TEST:
    phase4_path = OUTPUT_ROOT / "phase4_rrf_selection.json"
    phase3_selection_path = PHASE3_OUTPUT_ROOT / "phase3_model_selection.json"
    final_manifest_path = PHASE3_OUTPUT_ROOT / "final_model" / "training_manifest.json"
    assert phase4_path.exists() and phase3_selection_path.exists()
    assert final_manifest_path.exists() and FINAL_DENSE_CHECKPOINT.exists()
    selected = json.loads(phase4_path.read_text(encoding="utf-8"))
    phase3_selected = json.loads(phase3_selection_path.read_text(encoding="utf-8"))
    final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
    assert selected["hybrid_beats_both_components"] is True
    assert selected["full_validation"] is True
    assert selected["official_test_open_allowed"] is True
    assert selected["component_depth"] == COMPONENT_DEPTH
    assert selected["output_k"] == OUTPUT_K
    assert selected["tfidf_weight"] == TFIDF_WEIGHT
    assert selected["dense_weight"] == DENSE_WEIGHT
    assert selected["primary_metric"] == PRIMARY_SELECTION_METRIC
    assert tuple(selected["tie_breakers"]) == SELECTION_TIE_BREAKERS
    assert phase3_selected["winner"] == "fine_tuned"
    assert phase3_selected["official_test_open_allowed"] is True
    assert final_manifest["scope"] == "official_train"
    assert final_manifest["used_official_test_pairs"] is False
    selected_c = int(selected["selected_rrf_c"])

    dense_fingerprint = checkpoint_fingerprint(FINAL_DENSE_CHECKPOINT)
    dense_model = SentenceTransformer(str(FINAL_DENSE_CHECKPOINT))
    summaries = []
    for project, data in datasets.items():
        print(f"\n=== PHASE 4 OFFICIAL TEST: {project} ===")
        queries = query_sets[project]["test"]
        allowed_ids = data["official_train_ids"] | data["official_test_ids"]
        matrix, _ = fit_hybrid_tfidf(
            data["issues"], query_sets[project]["official_train_ids"]
        )
        tfidf_rankings = retrieve_sparse_rolling(
            data, queries, matrix, allowed_ids, COMPONENT_DEPTH
        )
        del matrix
        gc.collect()
        embeddings = encode_checkpoint_for_project(
            dense_model, data, "final_finetuned", dense_fingerprint
        )
        dense_rankings = retrieve_dense_rolling(
            data, queries, embeddings, allowed_ids, COMPONENT_DEPTH
        )
        tfidf_rankings.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_official_test_tfidf_top50.csv",
            index=False,
        )
        dense_rankings.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_official_test_dense_top50.csv",
            index=False,
        )
        fused = fuse_project(
            tfidf_rankings, dense_rankings, queries, data, selected_c
        )
        fused.to_csv(
            OUTPUT_ROOT / f"{project.lower()}_official_test_rrf_queries.csv",
            index=False,
        )
        tfidf_scored = score_component(tfidf_rankings, queries, "tfidf")
        dense_scored = score_component(dense_rankings, queries, "fine_tuned_dense")
        summaries.extend([
            summarize(tfidf_scored, "tfidf"),
            summarize(dense_scored, "fine_tuned_dense"),
            summarize(fused, "rrf_hybrid", selected_c),
        ])

    project_test_summary = pd.DataFrame(summaries)
    macro_rows = []
    for (system, rrf_c), group in project_test_summary.groupby(
        ["system", "rrf_c"], dropna=False, sort=True
    ):
        macro_rows.append(macro_row(group, system, rrf_c))
    official_test_summary = pd.concat(
        [project_test_summary, pd.DataFrame(macro_rows)], ignore_index=True
    )
    display(official_test_summary)
    official_test_summary.to_csv(OUTPUT_ROOT / "official_test_summary.csv", index=False)
else:
    print("OFFICIAL TEST SEALED.")


This model was created with Sentence Transformers version 5.6.0, but you're using version 5.4.1. Consider updating to the latest version to avoid potential issues.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== PHASE 4 OFFICIAL TEST: Firefox ===
  Firefox / TF-IDF: 500/4236
  Firefox / TF-IDF: 1000/4236
  Firefox / TF-IDF: 1500/4236
  Firefox / TF-IDF: 2000/4236
  Firefox / TF-IDF: 2500/4236
  Firefox / TF-IDF: 3000/4236
  Firefox / TF-IDF: 3500/4236
  Firefox / TF-IDF: 4000/4236
TF-IDF retrieval seconds: 2057.79


Batches:   0%|          | 0/905 [00:00<?, ?it/s]

  Firefox / dense: 256/4236
  Firefox / dense: 512/4236
  Firefox / dense: 768/4236
  Firefox / dense: 1024/4236
  Firefox / dense: 1280/4236
  Firefox / dense: 1536/4236
  Firefox / dense: 1792/4236
  Firefox / dense: 2048/4236
  Firefox / dense: 2304/4236
  Firefox / dense: 2560/4236
  Firefox / dense: 2816/4236
  Firefox / dense: 3072/4236
  Firefox / dense: 3328/4236
  Firefox / dense: 3584/4236
  Firefox / dense: 3840/4236
  Firefox / dense: 4096/4236
  Firefox / dense: 4236/4236
Dense retrieval seconds: 2.92

=== PHASE 4 OFFICIAL TEST: MozillaCore ===
  MozillaCore / TF-IDF: 500/4115
  MozillaCore / TF-IDF: 1000/4115
  MozillaCore / TF-IDF: 2000/4115
TF-IDF retrieval seconds: 2950.5


Batches:   0%|          | 0/1603 [00:00<?, ?it/s]

  MozillaCore / dense: 256/4115
  MozillaCore / dense: 512/4115
  MozillaCore / dense: 768/4115
  MozillaCore / dense: 1024/4115
  MozillaCore / dense: 1280/4115
  MozillaCore / dense: 1536/4115
  MozillaCore / dense: 1792/4115
  MozillaCore / dense: 2048/4115
  MozillaCore / dense: 2304/4115
  MozillaCore / dense: 2560/4115
  MozillaCore / dense: 2816/4115
  MozillaCore / dense: 3072/4115
  MozillaCore / dense: 3328/4115
  MozillaCore / dense: 3584/4115
  MozillaCore / dense: 3840/4115
  MozillaCore / dense: 4096/4115
  MozillaCore / dense: 4115/4115
Dense retrieval seconds: 6.28


,project,system,rrf_c,queries,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10,union_hit@50
0,Firefox,tfidf,NaN,4236,0.257318,0.440038,0.515817,0.128187,0.244808,0.299741,0.334305,0.277749,NaN
1,Firefox,fine_tuned_dense,NaN,4236,0.409585,0.633617,0.709632,0.211748,0.402559,0.477160,0.505686,0.435872,NaN
2,Firefox,rrf_hybrid,10.0,4236,0.370397,0.617092,0.703022,0.187288,0.379471,0.460920,0.474936,0.411135,0.868980
3,MozillaCore,tfidf,NaN,4115,0.215796,0.351397,0.409478,0.163124,0.285052,0.342154,0.273466,0.269433,NaN
4,MozillaCore,fine_tuned_dense,NaN,4115,0.307655,0.481652,0.546537,0.237596,0.399347,0.463059,0.380840,0.374533,NaN
5,MozillaCore,rrf_hybrid,10.0,4115,0.295747,0.483597,0.558931,0.227266,0.404565,0.479287,0.375949,0.375987,0.730498
6,MACRO_PROJECT_AVERAGE,fine_tuned_dense,NaN,8351,0.358620,0.557635,0.628084,0.224672,0.400953,0.470110,0.443263,0.405203,NaN
7,MACRO_PROJECT_AVERAGE,rrf_hybrid,10.0,8351,0.333072,0.550344,0.630976,0.207277,0.392018,0.470103,0.425442,0.393561,0.799739
8,MACRO_PROJECT_AVERAGE,tfidf,NaN,8351,0.236557,0.395718,0.462647,0.145655,0.264930,0.320948,0.303886,0.273591,NaN


In [12]:
from pathlib import Path

p = Path("/kaggle/working/phase4_rrf_outputs/official_test_summary.csv")

print("Exists:", p.exists())

if p.exists():
    import pandas as pd
    display(pd.read_csv(p))

Exists: True


,project,system,rrf_c,queries,hit@1,hit@5,hit@10,recall@1,recall@5,recall@10,mrr@10,ndcg@10,union_hit@50
0,Firefox,tfidf,NaN,4236,0.257318,0.440038,0.515817,0.128187,0.244808,0.299741,0.334305,0.277749,NaN
1,Firefox,fine_tuned_dense,NaN,4236,0.409585,0.633617,0.709632,0.211748,0.402559,0.477160,0.505686,0.435872,NaN
2,Firefox,rrf_hybrid,10.0,4236,0.370397,0.617092,0.703022,0.187288,0.379471,0.460920,0.474936,0.411135,0.868980
3,MozillaCore,tfidf,NaN,4115,0.215796,0.351397,0.409478,0.163124,0.285052,0.342154,0.273466,0.269433,NaN
4,MozillaCore,fine_tuned_dense,NaN,4115,0.307655,0.481652,0.546537,0.237596,0.399347,0.463059,0.380840,0.374533,NaN
5,MozillaCore,rrf_hybrid,10.0,4115,0.295747,0.483597,0.558931,0.227266,0.404565,0.479287,0.375949,0.375987,0.730498
6,MACRO_PROJECT_AVERAGE,fine_tuned_dense,NaN,8351,0.358620,0.557635,0.628084,0.224672,0.400953,0.470110,0.443263,0.405203,NaN
7,MACRO_PROJECT_AVERAGE,rrf_hybrid,10.0,8351,0.333072,0.550344,0.630976,0.207277,0.392018,0.470103,0.425442,0.393561,0.799739
8,MACRO_PROJECT_AVERAGE,tfidf,NaN,8351,0.236557,0.395718,0.462647,0.145655,0.264930,0.320948,0.303886,0.273591,NaN


## 9. Manifest and leakage assertions

In [13]:
run_manifest = {
    "phase": 4,
    "method": "equal-weight reciprocal rank fusion",
    "components": ["hybrid_word_char_tfidf", "phase3_fine_tuned_minilm"],
    "component_depth": COMPONENT_DEPTH,
    "output_k": OUTPUT_K,
    "rrf_c_grid": list(RRF_C_GRID),
    "weights": {"tfidf": TFIDF_WEIGHT, "dense": DENSE_WEIGHT},
    "selection_rule": {
        "aggregation": SELECTION_AGGREGATION,
        "primary_metric": PRIMARY_SELECTION_METRIC,
        "tie_breakers": list(SELECTION_TIE_BREAKERS),
    },
    "run_validation_component_retrieval": RUN_VALIDATION_COMPONENT_RETRIEVAL,
    "run_rrf_validation": RUN_RRF_VALIDATION,
    "run_official_test": RUN_OFFICIAL_TEST,
    "neural_training_in_phase4": False,
    "environment": {
        "python": sys.version.split()[0], "platform": platform.platform(),
        "torch": torch.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    "projects": {
        project: {
            "issues": len(data["issues"]),
            "validation_queries": len(query_sets[project]["validation"]),
            "test_queries": len(query_sets[project]["test"]),
        }
        for project, data in datasets.items()
    },
}
(OUTPUT_ROOT / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2), encoding="utf-8"
)
print(json.dumps(run_manifest, indent=2))
print("PHASE 4 MANIFEST WRITTEN")


{
  "phase": 4,
  "method": "equal-weight reciprocal rank fusion",
  "components": [
    "hybrid_word_char_tfidf",
    "phase3_fine_tuned_minilm"
  ],
  "component_depth": 50,
  "output_k": 10,
  "rrf_c_grid": [
    10,
    30,
    60,
    100
  ],
  "weights": {
    "tfidf": 1.0,
    "dense": 1.0
  },
  "selection_rule": {
    "aggregation": "macro_project_average",
    "primary_metric": "hit@10",
    "tie_breakers": [
      "mrr@10",
      "recall@10"
    ]
  },
  "run_validation_component_retrieval": false,
  "run_rrf_validation": false,
  "run_official_test": true,
  "neural_training_in_phase4": false,
  "environment": {
    "python": "3.12.13",
    "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
    "torch": "2.10.0+cu128",
    "sentence_transformers": "5.4.1",
    "cuda_available": true
  },
  "projects": {
    "Firefox": {
      "issues": 115814,
      "validation_queries": 3060,
      "test_queries": 4236
    },
    "MozillaCore": {
      "issues": 205069,
      "validation

## Interpretation

A successful Phase 4 result requires more than a raw increase on one project:

```text
macro-project hybrid Hit@10 > best component Hit@10
```

with MRR@10 and Recall@10 as frozen tie-breakers. Report the `dense_only` and
`tfidf_only` paired counts as direct evidence of complementarity, and report
`union_hit@50` as the maximum query coverage available to an RRF operating on
these two Top-50 lists.

If RRF does not beat fine-tuned MiniLM, report that result honestly. It means the
lexical ranking contains complementary hits but equal-weight rank fusion may not
place them effectively in Top-10; it does not invalidate Phases 1–3.
